## Vector Search

In [1]:
# Imports the utility to load raw text files into LangChain Document objects
from langchain_community.document_loaders import TextLoader 
# Imports the tool to break long documents into smaller, manageable chunks
from langchain_text_splitters import CharacterTextSplitter 
# Imports the interface to convert text into numerical vectors using OpenAI's models
from langchain_openai import OpenAIEmbeddings
# Imports the vector database used to store and efficiently search text embeddings             
from langchain_chroma import Chroma                       

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import pandas as pd

books_df = pd.read_csv("../data/processed/book_cleaned.csv")
books_df.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitles,tagged_description
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."


In [4]:
books_df["tagged_description"].head()

0    9780002005883 A NOVEL THAT READERS and critics...
1    9780002261982 A new 'Christie for Christmas' -...
2    9780006178736 A memorable, mesmerizing heroine...
3    9780006280897 Lewis' work on the nature of lov...
4    9780006280934 "In The Problem of Pain, C.S. Le...
Name: tagged_description, dtype: str

In [5]:
# opening a txt for writing the tagged descriptions
with open("../data/processed/tagged_description.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(books_df["tagged_description"].astype(str)))

In [6]:
raw_docs = TextLoader("../data/processed/tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100, separator="\n")
documents = text_splitter.split_documents(raw_docs)

Created a chunk of size 1168, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1088, which is longer than the specified 1000
Created a chunk of size 1189, which is longer than the specified 1000


Created a chunk of size 1253, which is longer than the specified 1000
Created a chunk of size 2006, which is longer than the specified 1000
Created a chunk of size 1222, which is longer than the specified 1000
Created a chunk of size 1184, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 1057, which is longer than the specified 1000
Created a chunk of size 1270, which is longer than the specified 1000
Created a chunk of size 1635, which is longer than the specified 1000
Created a chunk of size 1128, which is longer than the specified 1000
Created a chunk of size 1319, which is longer than the specified 1000
Created a chunk of size 1119, which is longer than the specified 1000
Created a chunk of size 1189, which is longer than the specified 1000
Created a chunk of size 2008, which is longer than the specified 1000
Created a chunk of s

In [7]:
documents[0]

Document(metadata={'source': '../data/processed/tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of 

In [8]:
db_books = Chroma.from_documents(documents, OpenAIEmbeddings())

In [9]:
query = "A Book to teach children about the importance of kindness"

docs = db_books.similarity_search(query, k=10)
docs

[Document(id='72f3dbec-7cea-4dcf-ac4a-ae9a4986256b', metadata={'source': '../data/processed/tagged_description.txt'}, page_content='9780590956161 When a new boy in his second grade class tries to get the other students to play a game that involves saying the meanest things possible to one another, Little Bill shows him a better way to make friends.\n9780590975001 Summary: Kidnapped from her orphanage by a BFG (Big Friendly Giant), who spends his life blowing happy dreams to children, Sophie concocts with him a plan to save the world from nine other man-gobbling cannibal giants.\n9780590988490 When her father is murdered and the police obtain an e-mail confession that implicates her, Alice Robie realizes she will have to flee and prove her innocence. Original.'),
 Document(id='144869f5-53a6-40eb-9b7f-fbdaf791b144', metadata={'source': '../data/processed/tagged_description.txt'}, page_content="9780941807555 THE LITTLE BIG BOOK FOR GOD'S CHILDREN is a wonderful resource for parents lookin

In [11]:
books_df[books_df["isbn13"]==int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitles,tagged_description
2870,9780590956161,0590956167,The Meanest Thing to Say,NaN,Bill Cosby;Varnette P. Honeywood,Juvenile Fiction,http://books.google.com/books/content?id=BCaaN...,When a new boy in his second grade class tries...,1997.0,3.53,40.0,956.0,The Meanest Thing to Say,9780590956161 When a new boy in his second gra...


In [12]:
# Creating a function that perform search and pull book details
def retrive_semantic_recomendations(query: str, k: int = 10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=k)
    book_list = []

    for  i in range(0, len(recs)):
        book_list += [int(recs[i].page_content.split()[0])]

    return books_df[books_df["isbn13"].isin(book_list)]


In [13]:
retrive_semantic_recomendations("a book about pirates")

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitles,tagged_description
198,9780060733353,0060733357,The Confusion,Volume Two of The Baroque Cycle,Neal Stephenson,Fiction,http://books.google.com/books/content?id=NJbll...,"In the year 1689, a cabal of Barbary galley sl...",2005.0,4.25,815.0,18790.0,The Confusion: Volume Two of The Baroque Cycle,"9780060733353 In the year 1689, a cabal of Bar..."
1074,9780237525378,0237525372,Oliver Twist,NaN,Pauline Francis;Charles Dickens,Juvenile Nonfiction,http://books.google.com/books/content?id=X6RvT...,This wonderful series is a quick way into a ra...,2003.0,3.66,48.0,92.0,Oliver Twist,9780237525378 This wonderful series is a quick...
2066,9780425205594,0425205592,Dark Watch,NaN,Clive Cussler;Jack B. Du Brul,Fiction,http://books.google.com/books/content?id=EKDrH...,When Japanese shipping magnates whose fortunes...,2005.0,4.09,357.0,5749.0,Dark Watch,9780425205594 When Japanese shipping magnates ...
3539,9780755323012,0755323017,The Death Ship of Dartmouth,NaN,Michael Jecks,Fiction,http://books.google.com/books/content?id=yVArA...,When a body is found lying in the middle of th...,2006.0,4.07,352.0,7.0,The Death Ship of Dartmouth,9780755323012 When a body is found lying in th...
3967,9780812552843,0812552849,The Fourth Book of Lost Swords,Farslayer's Story,Fred Saberhagen,Fiction,http://books.google.com/books/content?id=B_hiW...,"Black Pearl, a mermaid, Cosmos, her treacherou...",1990.0,3.83,252.0,1736.0,The Fourth Book of Lost Swords: Farslayer's Story,"9780812552843 Black Pearl, a mermaid, Cosmos, ..."
4440,9781402714573,1402714572,Treasure Island,NaN,Robert Louis Stevenson,Juvenile Fiction,http://books.google.com/books/content?id=UUl7Z...,While going through the possessions of a decea...,2004.0,3.83,213.0,393.0,Treasure Island,9781402714573 While going through the possessi...
4444,9781402726002,1402726007,The Adventures of Huckleberry Finn,NaN,Mark Twain,Juvenile Fiction,http://books.google.com/books/content?id=pi1gC...,"A nineteenth-century boy, floating down the Mi...",2006.0,3.81,305.0,350.0,The Adventures of Huckleberry Finn,"9781402726002 A nineteenth-century boy, floati..."
4507,9781416535522,1416535527,Santa Cruise,A Holiday Mystery at Sea,Mary Higgins Clark;Carol Higgins Clark,Fiction,http://books.google.com/books/content?id=QsbYE...,Embarking on a Mystery Seminar luxury cruise d...,2006.0,3.41,261.0,4631.0,Santa Cruise: A Holiday Mystery at Sea,9781416535522 Embarking on a Mystery Seminar l...
4545,9781423100188,1423100182,Pirates of the Caribbean: The Coming Storm - J...,Junior Novel,Rob Kidd,Juvenile Fiction,http://books.google.com/books/content?id=UkvjA...,Teenage stowaway Jack Sparrow and his band of ...,2006.0,3.83,135.0,1364.0,Pirates of the Caribbean: The Coming Storm - J...,9781423100188 Teenage stowaway Jack Sparrow an...
4550,9781423101680,1423101685,Pirates of the Caribbean: Age of Bronze - Jack...,NaN,Rob Kidd,Juvenile Fiction,http://books.google.com/books/content?id=iHKIN...,The beginning of a new story arc! Jack and com...,2006.0,4.05,144.0,532.0,Pirates of the Caribbean: Age of Bronze - Jack...,9781423101680 The beginning of a new story arc...
